# Noms du binome: 

FERNANDES--STRAG, TORRES--LEGUET

# Prénoms du binome:

Henry, Alexandre

## Note: exporter le compte rendu basé sur le notebook au format pdf


# Entropie et génération de mots de passe
* L'objectif de ce tp est de continuer à se familiariser avec la notion d'entropie, mais aussi de comprendre le lien qu'il existe entre cette mesure informationelle et la sécurité d'un générateur (humain ou executable) de mots de passes
* Ainsi, nous proposons d'étudier l'entropie d'un tel générateur, et ce en fonction du modèle probabiliste considéré pour le modéliser (contruit à partir d'une lettre, de deux lettres, de 4 lettres, ...). A l'aide de tirrages aléatoires, nous estimerons également le temps moyen nécessaire pour trouver un mot de passe à partir de ce modèle.
* A la fin de ce TP, nous considérerons un générateur de mots de passe spécifique qui générera un mot de passe en prennant **4 lettres consécutives dans un texte** (sans se soucier des espaces). Ces lettres peuvent faire parti d'un ou de plusieurs mots consécutifs.
* Nous faisons l'hypothèse que le texte n'est composé que des 26 lettres de l'alphabet, sans majuscules ni accents

Nous chercherons aussi à comprendre (voir dernière question):
- les bonnes pratiques pour le défenseur, i.e. la personne cherchant à générer/construire un système de génération de mots de passe.
- les bonnes pratiques pour l'attaquant, i.e. la personne essayant de trouver le mot de passe.

**Il est important de commenter vos réponses, en utilisant des cellules markdown**


In [1]:
import numpy as np

from numpy import genfromtxt
from pandas import read_csv
import pandas as pd
import time

#### Modèle monogramme (une lettre) : le générateur génère des mots de passe à partir des occurences des monogrammes
* On récupére des données composées de [lettre,frequence d'apparition de la lettre] (voir fichier csv pour [comma-separated-value](https://en.wikipedia.org/wiki/Comma-separated_values)) 

In [3]:
monogramme = read_csv('monogramme.csv')
freq_mono = (monogramme['frequency']).values
letters_mono = (monogramme['letters']).values

Q: Quelles sont les 5 lettres les plus représentées ?

In [5]:
letters_mono[0:5]

<StringArray>
['E', 'S', 'A', 'N', 'T']
Length: 5, dtype: str

On remarque que le fichier 'monogramme.csv' a déjà trié les lettres par leur fréquence.
Il suffit donc juste de récupérer les 5 premiers éléments de la liste lettters_mono : E,S,A,N,T.
Cela correspond bien aux lettres les plus fréquentes dans un texte écrit en français. 

Ecrire une fonction qui calcule l'entropie à partir d'un vecteur constitué de probabilités empiriques (note, il est important de bien *gérer* le cas ou la probabilité est nulle).

In [22]:
def entropie(freq):
    ent = 0
    for f in freq:
        if f > 0 and f < 1:
            ent += f * np.log2(f)
    return -ent

In [25]:
entropie([0.95, 0.05, 0])

np.float64(0.28639695711595625)

Q: en utilisant ce modèle probabiliste pour générer un mot de passe, quelle est l'entropie d'un mot de passe de 8 lettres ?

In [52]:
ent = entropie(freq_mono)
ent_totale = ent * 8

print(ent)
print(ent_totale)

3.959530303722292
31.676242429778338


Pour un mot de passe de 8 lettres généré en tirant aléatoirement et indépendamment des lettres selon la distribution lettres_mono, on peut simplement multiplier par 8 l'entropie de la distribution du monogramme, grâce à la propriété :
$H(X,Y) = H(X) + H(Y)$ où $X$ et $Y$ sont indépendantes (ici, les variables aléatoires indépendantes sont les tirages de chaque lettre). On obtient ainsi une entropie de 31.67.

Cela revient à tirer au hasard et uniformément un mot de passe parmi 2^(31.68) valeurs possibles, soit environ 2 milliards. (ie, la distribution de probabilité uniforme sur un ensemble de 2 milliards de valeurs à cette même entropie de 31.67)

Q: A l'aide de la fonction `np.random.choice()`, estimer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ? (note: ici le tirage n'est pas forcemment réaliste, car aléatoire, mais l'idée est surtout de mesurer le temps minimal nécessaire pour générer N mots de passes).

In [53]:
np.random.choice(letters_mono, 10, p=freq_mono)

array(['E', 'S', 'D', 'S', 'R', 'I', 'I', 'T', 'L', 'G'], dtype=object)

In [61]:
times = 0
N = 100
N_pwd = 100000

for _ in range(N):
    t = time.time()
    for _ in range(8):
        np.random.choice(letters_mono, N_pwd, p=freq_mono)
    t_100000 = time.time() - t
    times += t_100000

print(times/N, 's')

0.01444110631942749 s


On peut générer 100 000 mots de passe en beaucoup moins d'une seconde sur un ordinateur portable.

Nous definissons l'"entropie du devin" G (guessing entropie) comme le **nombre moyen d'essais successif nécessaires pour trouver un mot de passe à partir de notre générateur**.
    On peut montrer que $G\geq 2^H/4+1$ où $H$ est l'entropie de la source (voir le papier Password_Entropy_and_Password_Quality.pdf )

Q: calculer le minorant de $G$ pour ce modèle

In [60]:
inf = np.pow(2, ent_totale) / 4 + 1

print(inf)

857904864.6814479


On obtient une borne basse qui vaut 860 millions.
Cela veut dire que le nombre moyens d'essais successifs pour trouvre le mot de passe est plus grand que 857M.

On peut comparer ce nombre avec le nombre de 2 milliards trouvé plus haut (équivalent à l'entropie de 31.68) : c'est environ la moitié.

Q: combien de temps cela prendra-t-il pour trouver un mot de passe si l'on suppose qu'il est possible de prendre le générateur codé précédemment ? (en minutes)

In [68]:
time_per_pwd = (times/N) / N_pwd

print(time_per_pwd, 's')
print(time_per_pwd * inf, 's')
print(time_per_pwd * inf / 60, 'min')

1.444110631942749e-07 s
123.89095362818841 s
2.0648492271364733 min


La borne basse est donc de 124 seconde. Cela veut dire qu'en quelques minutes de recherche, on peut tomber sur le mot de passe.

On propose maintenant d'utiliser un modèle plus évolué qui est construit à partir de la probabilité conjointe de deux lettres successives (bigramme)

In [83]:
bigramme = read_csv('bigramme.csv',keep_default_na=False)

#print(bigramme.head(5))
freq_bi = (bigramme['frequency']).values

letters_bi = (bigramme['letters']).values

Q: Quelles sont les 5 couples de lettres les plus représentés ?

In [84]:
bigramme.sort_values(by='frequency', ascending=False, inplace=True)
bigramme.head(5)

,letters,frequency
122,ES,0.023809
117,EN,0.021248
82,DE,0.019570
290,LE,0.018845
357,NT,0.017009


Q: en utilisant ce modèle probabiliste pour générer un mot de passe, quelle est l'entropie d'un mot de passe de 8 lettres ?

In [89]:
ent = entropie(freq_bi)
ent_totale = 4 * ent

print(ent)
print(ent_totale)

7.535566011616047
30.142264046464188


On utilise la même méthode de calculs que précédemment, en sommant les entropies des variables indépendantes (ici, le tirage d'un bigramme)

Q: Pourquoi cette entropie est-elle inférieure à celle du modèle construit sur des monogrammes ? Quelle propriété théorique de l'entropie peut justifier ce constat  ?

La baisse d'entropie est expliquée par cette formule :

$$
H(X+Y) = H(X) + H(Y) - I(X+Y)
$$

qui, appliquée à nos 8 variables :

$$
H(X_1+...+X_8) = \Sigma_i H(X_i) - \Sigma_{i \neq j} I(H_i, H_j)
$$

par construction des mots de passe, les informations mutuelles I(X_1, X_3), I(X_3,X_5) etc (ie, les lettres qui ne sont pas tirés en même temps) sont nulles. Il reste seulement :

$$
H(X_1+...+X_8) = \Sigma_i H(X_i) - \Sigma_{i,j \in \text{bigramme}} I(H_i, H_j)
$$

On voit donc que $H(X_1+...+X_8)$ est bien plus petit que $\Sigma_i H(X_i)$, et cela est dû au fait qu'on perd l'information mutuelle au sein des lettres tirées en même temps.


Q: A l'aide de la fonction `np.random.choice()`, calculer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ?

In [ ]:
times = 0
N = 100
N_pwd = 100000

for _ in range(N):
    t = time.time()
    for _ in range(4):
        np.random.choice(letters_bi, N_pwd, p=freq_bi)
    t_100000 = time.time() - t
    times += t_100000

print(times/N, 's')

0.017042951583862306 s


Q: calculer le minorant de $G$ pour ce modèle

In [93]:
inf = np.pow(2, ent_totale) / 4 + 1

print(inf)

296254956.70154107


Q: combien de temps cela prendra-t-il pour trouver un mot de passe si l'on suppose qu'il est possible de prendre le générateur codé précédemment ? (en minutes)

In [94]:
time_per_pwd = (times/N) / N_pwd

print(time_per_pwd, 's')
print(time_per_pwd * inf, 's')
print(time_per_pwd * inf / 60, 'min')

1.7042951583862306e-07 s
50.49058883543588 s
0.8415098139239313 min


Ce mot de passe est plus facile à trouver (ce qui est normal car son entropie est plus faible).

Q: **Modèle Uniforme:** si maintenant on change de stratégie et on tire aléatoirement chaque lettre de l'alphabet de façon uniforme, quelle est l'entropie de ce nouveau générateur ?

In [97]:
ent = entropie([1/26]*26)
ent_totale = 8 * ent

print(ent)
print(ent_totale)

4.70043971814109
37.60351774512872


On trouve 37.6. Comme on est dans le cas d'une distribution uniforme sur 26^8 valeurs, on pouvait aussi directement calculer log2(26^8)=37.6.

Q: A l'aide de la fonction `np.random.choice()`, calculer le temps nécessaire en secondes pour tirer 100 000 mots de passes en utilisant ce générateur ?

In [120]:
times = 0
N = 100
N_pwd = 100000
proba = [1/26]*26

for _ in range(N):
    t = time.time()
    for _ in range(8):
        np.random.choice(letters_mono, N_pwd, p=proba)
    t_100000 = time.time() - t
    times += t_100000

print(times/N, 's')

0.017895288467407226 s


Q: calculer le minorant de $G$ pour ce modèle

In [112]:
inf = np.pow(2, ent_totale) / 4 + 1

print(inf)

52206766144.99937


Q: dans ce cas précis, quelle est la valeur exacte de $G$?

In [162]:
G = np.pow(26, 8)

print(G)

208827064576


loi géométrique

Il y a 26^8 possibilités équiprobables, le nombre d'essais moyens pour trouver la bonne valeur est le nombre de possibilités divisé par 2.

Q: combien de temps cela prendra-t-il pour trouver un mot de passe en utilisant le générateur codé précédemment ? (en minutes)

In [124]:
time_per_pwd = (times/N) / N_pwd

print(time_per_pwd, 's')
print(time_per_pwd * G, 's')
print(time_per_pwd * G / 60, 'min')

1.7895288467407226e-07 s
18685.102801946985 s
311.41838003244976 min


Q: implémenter une attaque pratique qui consiste à:
1. **pour le défenseur: (la personne qui génère le mot de passe)** tirer un mot de passe de 4 lettres consécutives à partir de ce texte de Victor Hugo (texteFrancais.txt) tiré des Misérables.  
2. **pour l'attaquant:** utiliser le modèle bigramme pour générer des mots de passe et minimiser le nombre d'essais. Pour cela on pourra :
    * dans un premier temps pré-calculer un **dictionnaire**, qui contriendra un nombre de MdP générés classés dans l'ordre du plus probable au moins probable et qui ne contient pas de doublons 
    * dans un deuxième temps appeler ce dictionnaire pour comparer chacune de ses entrées au mot de passe généré.
3. Il faudra faire ses tests plusieurs fois afin de d'obtenir un nombre moyens d'appel au dictionnaire nécessaire
4. Il sera intéressant de comparer le nombre trouvé à la valeur de G (qui est une borne inférieure)
5. Question annexe: Par un simple calcul, si le générateur utilisé n'est plus ce générateur mais un générateur qui tire chaque lettre de façon équiprobable, rappeler la valeur de G. Comparer cette valeur avec la valeur trouvée en utilisant la stratégie "des 4 lettres consécutives".

In [ ]:
## Fonction générant un mot de passe
def get_passwd():
    text_hugo = open("texteFrancais.txt","r")
    str_hugo = str(text_hugo.read())

    # On remplace des lettres avec accent avec des lettres sans accent
    str_hugo = str_hugo.replace("Â", "A")
    str_hugo = str_hugo.replace("Ù", "U")
    str_hugo = str_hugo.replace("Ô", "O")
    size_txt = len(str_hugo)

    idx_rand = np.random.randint(size_txt-4)
    #print(idx_rand)

    psswd = str_hugo[idx_rand:idx_rand+4]
    return(psswd)

In [186]:
N_dict = 1_000_000
proba_bi = freq_bi

# on génère N_dict mot de passe de 4 lettres à partir des b1 et b2
b1 = np.random.choice(letters_bi, N_dict, p=proba_bi).astype(str)
b2 = np.random.choice(letters_bi, N_dict, p=proba_bi).astype(str)
# qu'on place dans une liste (np.char.add pour concat des tableaux de string)
tab_passwd = np.char.add(b1, b2)

print(tab_passwd)

['TFON' 'TIIL' 'UIAS' ... 'ASUN' 'IBPS' 'NATC']


In [192]:
# on récupère la liste des mdps uniques et leur nombre (pour estimer leur proba)
mdps, counts = np.unique(tab_passwd, return_counts=True)

ordre = np.argsort(-counts) # indices des mdps triés par ordre décroissant de fréquence

dico = mdps[ordre]
freq_dico = counts[ordre] / N_dict # calcule de la proba empirique

print(dico)
print(freq_dico)

# une autre approche est de considérer tous les couples de bigrammes possibles (676^2)
# pour chacun des couples, on peut calculer leur probabilité grâce à proba_bi
# mais cette méthode n'est plus envisageable avec des modèles trigrammes, etc, alors que la présente méthode oui

['ESES' 'ENES' 'ESEN' ... 'IMSB' 'IMOH' 'ZYVI']
[6.00e-04 5.29e-04 5.15e-04 ... 1.00e-06 1.00e-06 1.00e-06]


In [193]:
pd.DataFrame({'mdp': dico[:10], 'frequence': freq_dico[:10]})

,mdp,frequence
0,ESES,0.000600
1,ENES,0.000529
2,ESEN,0.000515
3,ENEN,0.000468
4,ESDE,0.000467
5,DEES,0.000459
6,NTES,0.000459
7,LEES,0.000458
8,ENDE,0.000430
9,REES,0.000427


In [191]:
# rang de chaque mot dans le dictionnaire (1 = premier essai), pour une recherche en O(1)
#rang = {m: i + 1 for i, m in enumerate(dico)}

In [ ]:
NB_ATTAQUES = 1000
essais_moyens = 0

for _ in range(NB_ATTAQUES):
    pwd = get_passwd() # mot de passe caché, à trouver

    essais = 0
    for i, m in enumerate(dico): # dico est rangé du plus probable au moins probable
        essais += 1
        if m == pwd:
            break

    essais_moyens += essais

print(essais_moyens / NB_ATTAQUES)

13386.033


Le nombre d'essais moyen (ou de façon équivalente le nombre d'appels au dictionnaire) est d'environ 13k.
Par ailleurs, pour un attaquant naif, la borne inf de G vaut ici :

In [227]:
ent = 2 * entropie(freq_bi)
G_inf = np.power(2, ent) / 4 + 1

print(ent)
print(G_inf)

15.071132023232094
8607.029219412707


In [27]:
# Attaques sur 1000 mots de passes

nb_trial = 1000
vec_nb_trials = np.zeros(nb_trial)
trouve = np.zeros(nb_trial, dtype=bool)

for k in range(nb_trial):
    pwd = get_passwd()               # le défenseur tire 4 lettres consécutives du texte
    if pwd in rang:                  # l'attaquant parcourt son dictionnaire dans l'ordre
        vec_nb_trials[k] = rang[pwd]
        trouve[k] = True
    else:                            # mot absent du dictionnaire : attaque échouée
        vec_nb_trials[k] = len(dico)

print(f"Taux de succès : {100 * trouve.mean():.1f} %")
print(f"Nombre moyen d'essais (mdp trouvés) : {vec_nb_trials[trouve].mean():.0f}")
print(f"Nombre médian d'essais (mdp trouvés) : {np.median(vec_nb_trials[trouve]):.0f}")

Taux de succès : 99.4 %
Nombre moyen d'essais (mdp trouvés) : 13152
Nombre médian d'essais (mdp trouvés) : 4624


Le défenseur pense avoir un mot de passe « aléatoire » mais il est en fait tiré dans du texte français : sa distribution ressemble beaucoup à celle du modèle bigramme. L'attaquant le trouve donc presque toujours (les rares échecs sont des séquences très improbables, absentes du dictionnaire), et en quelques milliers d'essais seulement, soit une fraction de seconde avec le générateur chronométré plus haut.

La moyenne est tirée vers le haut par quelques mots de passe mal classés ; la médiane montre qu'un mot de passe sur deux est trouvé en encore moins d'essais.

In [28]:
# Comparaison avec la valeur de G

H_4 = 2 * entropie(proba_bi)           # entropie d'un mdp de 4 lettres (2 bigrammes)
G_bi = 2**H_4 / 4 + 1                  # borne inférieure de G
G_unif = (26**4 + 1) / 2               # G exact pour 4 lettres équiprobables

print(f"Entropie modèle bigramme (4 lettres) : {H_4:.2f} bits")
print(f"Borne inférieure de G (bigramme)     : {G_bi:.0f}")
print(f"Nombre moyen d'essais mesuré         : {vec_nb_trials[trouve].mean():.0f}")
print(f"G pour un générateur uniforme        : {G_unif:.0f}")
print(f"Rapport uniforme / mesuré            : {G_unif / vec_nb_trials[trouve].mean():.0f}")

Entropie modèle bigramme (4 lettres) : 15.07 bits
Borne inférieure de G (bigramme)     : 8607
Nombre moyen d'essais mesuré         : 13152
G pour un générateur uniforme        : 228488
Rapport uniforme / mesuré            : 17


**Comparaison avec la borne de G :** le nombre moyen d'essais mesuré est du même ordre de grandeur que la borne $2^H/4+1$, et lui est supérieur, ce qui est cohérent puisque c'est une borne inférieure. L'écart vient de deux choses : l'ordre de notre dictionnaire est seulement estimé par tirage, et le modèle bigramme n'est qu'une approximation de la vraie distribution des 4-grammes du texte (par exemple, un mot de passe peut commencer au milieu d'un bigramme).

**Question annexe :** avec un générateur qui tire chaque lettre de façon équiprobable, il y a $26^4 = 456\,976$ mots de passe équiprobables et $G = (26^4+1)/2 \approx 228\,488$. La stratégie « 4 lettres consécutives d'un texte » est donc environ 15 à 20 fois plus facile à casser, alors que les mots de passe ont la même longueur : ce qui compte est l'entropie du générateur, pas la longueur.

## Conclusions 

- Définir des bonnes pratiques pour le défenseur, i.e. la personne cherchant à concevoir un système de génération de mots de passe ? 
- Définir des bonnes pratiques pour l'attaquant, i.e. la personne essayant de trouver le mot de passe ?

**Bonnes pratiques pour le défenseur :**
- Utiliser un générateur réellement aléatoire et uniforme (par exemple le module `secrets`), pas un choix humain ni un extrait de texte : la structure de la langue fait chuter l'entropie.
- Augmenter la longueur et la taille de l'alphabet (minuscules, majuscules, chiffres, symboles) : l'entropie vaut $L \log_2(|A|)$ et G croît exponentiellement avec.
- Raisonner en entropie du *générateur* (et non en longueur apparente) et viser par exemple plus de 70 à 80 bits. Une phrase de passe composée de mots tirés au hasard dans une grande liste est un bon compromis.
- Ne pas réutiliser ses mots de passe, et côté serveur : utiliser un hachage lent et salé (bcrypt, argon2) et limiter le nombre d'essais.

**Bonnes pratiques pour l'attaquant :**
- Modéliser la distribution réelle des mots de passe (n-grammes de la langue, fuites de bases de données, mots courants) plutôt que d'attaquer en force brute uniforme.
- Tester les candidats du plus probable au moins probable, sans doublon : c'est la stratégie qui minimise le nombre moyen d'essais (et qui atteint G).
- Pré-calculer le dictionnaire une fois pour toutes (et le réutiliser sur de nombreuses cibles), puis le compléter avec des règles de transformation (majuscule initiale, chiffres à la fin, leet speak…).

## Un peu de lecture
Cet article montre comment des hackers, à partir de leaks de bases de mots de passes, peuvent rapidement arriver à trouver le votre:
https://arstechnica.com/information-technology/2013/05/how-crackers-make-minced-meat-out-of-your-passwords/
